# Cosmos DB → raw volume

Reads the `sales_orders` collection with **pymongo** (Serverless cannot use the
JVM Mongo connector) and lands it as JSON.

Nested fields — `ordered_products`, `clicked_items`, `promo_info` — are kept as
JSON *strings* here. Documents are not all the same shape, and the raw layer's
job is to preserve what arrived, not to interpret it. Silver parses them.

In [ ]:
%pip install -q pymongo==4.10.1

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from common_utils.ingestors import read_mongo_collection, write_files
from common_utils.logger import get_logger, log_info
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.settings import get_secret, load_json, parse_run_date
from common_utils.writers import create_namespace

In [ ]:
dbutils.widgets.text("config_path", "ingestion/config/cosmos_sales_orders.json")
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("raw_volume", "raw_data")
dbutils.widgets.text("secret_scope", "retail-platform-dev")
dbutils.widgets.text("run_date", date.today().isoformat())

config = load_json(dbutils.widgets.get("config_path"))
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
raw_volume = dbutils.widgets.get("raw_volume")
scope = dbutils.widgets.get("secret_scope")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
run_id = new_run_id()

connection = config["connection"]
target = f"/Volumes/{catalog}/{bronze_schema}/{raw_volume}/{config['source_name']}/load_date={run_date}"
logger = get_logger("ingestion")

In [ ]:
create_namespace(spark, catalog, bronze_schema, raw_volume, comment="Bronze: raw landing volume and raw copies of source data")
ensure_ops_schema(spark, catalog)

with track(spark, catalog, run_id, run_date, task="cosmos_ingestion", layer="raw", entity=config["source_name"]) as stats:
    log_info(logger, "connecting", database=connection["database"], collection=connection["collection"])

    df = read_mongo_collection(
        spark,
        connection_string=get_secret(dbutils, scope, config["secret_keys"]["connection_string"]),
        database=connection["database"],
        collection=connection["collection"],
        batch_size=connection.get("batch_size", 1000),
    )

    rows = df.count()
    write_files(df, target, config["landing_format"])
    stats.rows_read = stats.rows_written = rows
    log_info(logger, "landed", source=config["source_name"], rows=rows, path=target)

In [ ]:
display(spark.read.json(target).limit(5))